In [ ]:
import sys
sys.path.insert(0, r"C:\Users\20242732\Desktop\cbl\cbl_predictive_new\4cblw010\src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
df = pd.read_parquet(r"C:\Users\20242732\Desktop\cbl\cbl_predictive_new\4cblw010\src\cbl\data\merged.parquet")

print("Total samples:", len(df))
print("Columns:", df.columns.tolist())
print("\nClass counts:")
for col in ["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"]:
    print(f"  {col}: {df[col].sum()} ({df[col].sum()/len(df)*100:.2f}%)")

Total samples: 191872
Columns: ['smiles', 'wavenumber', 'transmittance', 'carboxylic_acid', 'amino', 'sulfonic_acid', 'guanidino', 'source']

Class counts:
  carboxylic_acid: 79005 (41.18%)
  amino: 116276 (60.60%)
  sulfonic_acid: 77 (0.04%)
  guanidino: 3145 (1.64%)


In [3]:
# Cell 3 - Sampling (keep all sulfonic acid, sample rest)
mask_sulfonic = df['sulfonic_acid'] == 1
df_rare   = df[mask_sulfonic]
df_common = df[~mask_sulfonic].sample(n=5000, random_state=42)
df_svm    = pd.concat([df_rare, df_common]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Subset size:", len(df_svm))
print("Sulfonic acid in subset:", df_svm['sulfonic_acid'].sum())
print("Guanidino in subset:    ", df_svm['guanidino'].sum())

Subset size: 5077
Sulfonic acid in subset: 77
Guanidino in subset:     100


In [4]:
# Cell 4 - Features, labels, split
X = np.stack(df_svm["transmittance"].values)
y = df_svm[["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"]].values

strata = df_svm['sulfonic_acid'].astype(str) + df_svm['guanidino'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strata
)

print("Train size:", len(X_train))
print("Test size: ", len(X_test))

Train size: 4061
Test size:  1016


In [5]:
# Cell 5 - Scale
trf = StandardScaler()
X_train_trf = trf.fit_transform(X_train)
X_test_trf  = trf.transform(X_test)

In [6]:
# Cell 6 - Baseline SVM (before tuning)
svm = MultiOutputClassifier(SVC(kernel='rbf', C=1.0, gamma='scale'))
svm.fit(X_train_trf, y_train)
y_pred = svm.predict(X_test_trf)

print("Baseline SVM (C=1, rbf, gamma=scale):")
print(classification_report(
    y_test, y_pred,
    target_names=["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"],
    zero_division=0
))

Baseline SVM (C=1, rbf, gamma=scale):
                 precision    recall  f1-score   support

carboxylic_acid       0.00      0.00      0.00       436
          amino       0.59      1.00      0.74       592
  sulfonic_acid       1.00      0.07      0.12        15
      guanidino       1.00      0.05      0.10        20

      micro avg       0.59      0.56      0.57      1063
      macro avg       0.65      0.28      0.24      1063
   weighted avg       0.36      0.56      0.42      1063
    samples avg       0.58      0.56      0.57      1063



In [7]:
# Cell 7 - Fix 1: add class_weight='balanced' to handle imbalance per sub-classifier
svm = MultiOutputClassifier(SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))
svm.fit(X_train_trf, y_train)
y_pred = svm.predict(X_test_trf)

print("Balanced SVM:")
print(classification_report(
    y_test, y_pred,
    target_names=["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"],
    zero_division=0
))

Balanced SVM:
                 precision    recall  f1-score   support

carboxylic_acid       0.43      1.00      0.60       436
          amino       0.59      1.00      0.74       592
  sulfonic_acid       0.12      0.07      0.09        15
      guanidino       1.00      0.05      0.10        20

      micro avg       0.51      0.97      0.67      1063
      macro avg       0.54      0.53      0.38      1063
   weighted avg       0.53      0.97      0.66      1063
    samples avg       0.51      0.97      0.66      1063



In [8]:
# Cell 8 - Fix 2: try linear kernel (often better for high-dim spectral data)
svm_linear = MultiOutputClassifier(SVC(kernel='linear', C=1.0, class_weight='balanced'))
svm_linear.fit(X_train_trf, y_train)
y_pred_linear = svm_linear.predict(X_test_trf)

print("Linear kernel SVM:")
print(classification_report(
    y_test, y_pred_linear,
    target_names=["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"],
    zero_division=0
))

Linear kernel SVM:
                 precision    recall  f1-score   support

carboxylic_acid       0.98      0.96      0.97       436
          amino       0.99      0.97      0.98       592
  sulfonic_acid       0.57      0.87      0.68        15
      guanidino       0.37      0.75      0.49        20

      micro avg       0.95      0.96      0.96      1063
      macro avg       0.72      0.89      0.78      1063
   weighted avg       0.97      0.96      0.96      1063
    samples avg       0.95      0.96      0.95      1063



In [9]:
# Cell 9 - Fix 3: dimensionality reduction before SVM (helps rbf kernel a lot)
from sklearn.decomposition import PCA

pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_trf)
X_test_pca  = pca.transform(X_test_trf)

print(f"Variance explained by 100 components: {pca.explained_variance_ratio_.sum():.3f}")

svm_pca = MultiOutputClassifier(SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))
svm_pca.fit(X_train_pca, y_train)
y_pred_pca = svm_pca.predict(X_test_pca)

print("\nRBF SVM + PCA(100):")
print(classification_report(
    y_test, y_pred_pca,
    target_names=["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"],
    zero_division=0
))

Variance explained by 100 components: 1.000

RBF SVM + PCA(100):
                 precision    recall  f1-score   support

carboxylic_acid       0.43      1.00      0.60       436
          amino       0.59      1.00      0.74       592
  sulfonic_acid       0.12      0.07      0.09        15
      guanidino       1.00      0.05      0.10        20

      micro avg       0.51      0.97      0.67      1063
      macro avg       0.54      0.53      0.38      1063
   weighted avg       0.53      0.97      0.66      1063
    samples avg       0.51      0.97      0.66      1063



In [10]:
# Cell 10 - Tune C for linear kernel
from sklearn.model_selection import GridSearchCV

param_grid = {'estimator__C': [0.01, 0.1, 1, 10, 100]}

svm_tuned = GridSearchCV(
    MultiOutputClassifier(SVC(kernel='linear', class_weight='balanced')),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

svm_tuned.fit(X_train_trf, y_train)

print(f"Best C: {svm_tuned.best_params_}")
print(f"Best CV score: {svm_tuned.best_score_:.4f}")

y_pred_tuned = svm_tuned.best_estimator_.predict(X_test_trf)
print(classification_report(
    y_test, y_pred_tuned,
    target_names=["carboxylic_acid", "amino", "sulfonic_acid", "guanidino"],
    zero_division=0
))

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best C: {'estimator__C': 10}
Best CV score: 0.9697
                 precision    recall  f1-score   support

carboxylic_acid       0.97      0.96      0.97       436
          amino       0.99      0.98      0.99       592
  sulfonic_acid       0.69      0.73      0.71        15
      guanidino       0.42      0.55      0.48        20

      micro avg       0.96      0.96      0.96      1063
      macro avg       0.77      0.81      0.78      1063
   weighted avg       0.96      0.96      0.96      1063
    samples avg       0.96      0.96      0.96      1063

